# Bài 24 - Pooling

Transformer Encoder tạo một vector cho mỗi token, `H` có dạng `(batch, sequence_length, hidden_dim)`. Mean pooling gộp chiều token thành một vector đại diện cho toàn câu. Cần bỏ qua padding bằng mask; vector sau pooling có dạng `(batch, hidden_dim)`.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

np.set_printoptions(precision=3, suppress=True)
tf.random.set_seed(42)
print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


## 1. Token representations và padding mask

Mỗi hàng là một câu; mỗi token có hidden vector 3 chiều trong ví dụ minh họa. Các vị trí cuối câu ngắn được padding. Mask bằng `True` cho token thật và `False` cho padding.

In [ ]:
# H minh họa output của Transformer Encoder, shape (batch, T, d).
H = np.array([
    [[1., 0., 1.], [3., 2., 1.], [99., 99., 99.], [99., 99., 99.]],
    [[2., 4., 0.], [0., 2., 2.], [4., 0., 2.], [99., 99., 99.]],
], dtype=np.float32)

valid_tokens = np.array([
    [True, True, False, False],
    [True, True, True, False],
])

print('H shape:', H.shape)
print('Mask shape:', valid_tokens.shape)
print('Số token hợp lệ mỗi câu:', valid_tokens.sum(axis=1))
print('H của câu thứ nhất:\n', H[0])
print('Mask của câu thứ nhất:', valid_tokens[0])

H shape: (2, 4, 3)
Mask shape: (2, 4)
Số token hợp lệ mỗi câu: [2 3]
H của câu thứ nhất:
 [[ 1.  0.  1.]
 [ 3.  2.  1.]
 [99. 99. 99.]
 [99. 99. 99.]]
Mask của câu thứ nhất: [ True  True False False]


## 2. Masked mean pooling

Với mỗi câu, cộng hidden vector của token hợp lệ rồi chia cho số token hợp lệ. Phép tính giảm chiều `T`, giữ chiều hidden `d`.

In [ ]:
def masked_mean_pooling(token_representations, mask):
    weights = mask[..., np.newaxis].astype(token_representations.dtype)
    token_sum = np.sum(token_representations * weights, axis=1)
    token_count = np.maximum(np.sum(weights, axis=1), 1.0)
    return token_sum / token_count

sentence_vectors = masked_mean_pooling(H, valid_tokens)
print('Sentence vectors (shape', sentence_vectors.shape, '):\n', sentence_vectors)
print('Số chiều đầu vào:', H.shape)
print('Số chiều sau pooling:', sentence_vectors.shape)

Sentence vectors (shape (2, 3) ):
 [[2.    1.    1.   ]
 [2.    2.    1.333]]
Số chiều đầu vào: (2, 4, 3)
Số chiều sau pooling: (2, 3)


Padding có vector lớn trong ví dụ để làm rõ tác dụng của mask. Nếu lấy trung bình trực tiếp trên mọi vị trí, padding làm thay đổi kết quả.

In [ ]:
unmasked_mean = H.mean(axis=1)
comparison = pd.DataFrame({
    'Câu': ['Câu 1 (2 token)', 'Câu 2 (3 token)'],
    'Mean có mask': [sentence_vectors[0].round(3).tolist(), sentence_vectors[1].round(3).tolist()],
    'Mean không mask': [unmasked_mean[0].round(3).tolist(), unmasked_mean[1].round(3).tolist()],
})
display(comparison)

,Câu,Mean có mask,Mean không mask
0,Câu 1 (2 token),"[2.0, 1.0, 1.0]","[50.5, 50.0, 50.0]"
1,Câu 2 (3 token),"[2.0, 2.0, 1.3329999446868896]","[26.25, 26.25, 25.75]"


## 3. Đóng gói thành Keras layer

Layer nhận token representations `(B, T, d)` và mask `(B, T)`. Có thể đặt layer sau Transformer Encoder và trước lớp phân loại.

In [ ]:
class MaskedMeanPooling(layers.Layer):
    def call(self, inputs):
        token_representations, mask = inputs
        mask = tf.cast(mask[..., tf.newaxis], token_representations.dtype)
        token_sum = tf.reduce_sum(token_representations * mask, axis=1)
        token_count = tf.maximum(
            tf.reduce_sum(mask, axis=1),
            tf.ones_like(tf.reduce_sum(mask, axis=1)),
        )
        return token_sum / token_count

keras_pooling = MaskedMeanPooling()([tf.constant(H), tf.constant(valid_tokens)])
print('Keras output shape:', keras_pooling.shape)
print('Keras pooled vectors:\n', keras_pooling.numpy())

Keras output shape: (2, 3)
Keras pooled vectors:
 [[2.    1.    1.   ]
 [2.    2.    1.333]]


## 4. Nối pooling với linear classifier

Sau pooling, mỗi câu có vector kích thước cố định `d`, bất kể độ dài ban đầu. Linear layer biến vector đó thành xác suất positive. Mô hình dưới đây minh họa cấu trúc sau Encoder; xác suất chưa có ý nghĩa đánh giá vì classifier chưa được huấn luyện.

In [ ]:
token_input = layers.Input(shape=(H.shape[1], H.shape[2]), name='encoder_token_vectors')
mask_input = layers.Input(shape=(H.shape[1],), dtype='bool', name='valid_token_mask')
pooled = MaskedMeanPooling(name='masked_mean_pooling')([token_input, mask_input])
probability = layers.Dense(1, activation='sigmoid', name='positive_probability')(pooled)
pooling_classifier = models.Model([token_input, mask_input], probability)
pooling_classifier.summary()

demo_probabilities = pooling_classifier([H, valid_tokens], training=False).numpy().ravel()
print('Output shape:', demo_probabilities.shape)
print('Demo probabilities (untrained classifier):', demo_probabilities.round(4))

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_token_vect… │ (None, 4, 3)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ valid_token_mask    │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masked_mean_pooling │ (None, 3)         │          0 │ encoder_token_ve… │
│ (MaskedMeanPooling) │                   │            │ valid_token_mask… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positive_probabili… │ (None, 1)         │          4 │ masked_mean_pool… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4 (16.00 B)

 Trainable params: 4 (16.00 B)

 Non-trainable params: 0 (0.00 B)

Output shape: (2,)
Demo probabilities (untrained classifier): [0.837 0.953]
